# Story-Voice Fine-Tuner – Step 2: Prepare Data
**Gradio UI version**

We will:
- Install all required libraries
- Upload your `my_stories.txt` file
- Clean, chunk, and split the text into train/validation sets
- Save the prepared dataset to disk

In [1]:
# ── Install all required libraries ──
!pip install -q transformers datasets accelerate evaluate gradio huggingface_hub

print("✅ All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
✅ All libraries installed!


In [2]:
from google.colab import files
import os

# ── Upload your writing samples file ──
print("📤 Upload your my_stories.txt file now...")
uploaded = files.upload()

txt_filename = list(uploaded.keys())[0]

# Save to a consistent location
os.makedirs("/content/data", exist_ok=True)
os.rename(txt_filename, "/content/data/my_stories.txt")

print(f"✅ Saved to /content/data/my_stories.txt")

📤 Upload your my_stories.txt file now...


Saving my_stories.txt to my_stories.txt
✅ Saved to /content/data/my_stories.txt


In [3]:
from transformers import GPT2Tokenizer
from datasets import Dataset

# ── Load and clean text ──
with open("/content/data/my_stories.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

raw_text = "\n".join(line.strip() for line in raw_text.splitlines() if line.strip())
print(f"✅ Loaded {len(raw_text):,} characters ({len(raw_text)/1024:.1f} KB)")

# ── Tokenizer ──
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# ── Chunk into 128-token blocks (creates many training examples) ──
BLOCK_SIZE = 128

def create_chunks(text):
    tokens = tokenizer.encode(text)
    return [tokens[i:i+BLOCK_SIZE] for i in range(0, len(tokens) - BLOCK_SIZE + 1, BLOCK_SIZE)]

# ── 90/10 train/validation split ──
split_index = int(len(raw_text) * 0.9)
train_chunks = create_chunks(raw_text[:split_index])
val_chunks   = create_chunks(raw_text[split_index:])

tokenized_train = Dataset.from_dict({
    "input_ids":      train_chunks,
    "attention_mask": [[1] * len(c) for c in train_chunks]
})
tokenized_val = Dataset.from_dict({
    "input_ids":      val_chunks,
    "attention_mask": [[1] * len(c) for c in val_chunks]
})

# ── Save to disk ──
tokenized_train.save_to_disk("/content/data/tokenized_train")
tokenized_val.save_to_disk("/content/data/tokenized_val")

total_steps = (len(tokenized_train) // 4) * 12
print(f"✅ Train chunks:   {len(tokenized_train):,}")
print(f"   Val chunks:     {len(tokenized_val):,}")
print(f"   Total steps:    ~{total_steps:,}  (~{total_steps * 0.8 / 60:.0f} minutes estimated)")
print("✅ Datasets saved to /content/data/")

✅ Loaded 1,446,679 characters (1412.8 KB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (303737 > 1024). Running this sequence through the model will result in indexing errors


Saving the dataset (0/1 shards):   0%|          | 0/2372 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/271 [00:00<?, ? examples/s]

✅ Train chunks:   2,372
   Val chunks:     271
   Total steps:    ~7,116  (~95 minutes estimated)
✅ Datasets saved to /content/data/


# Story-Voice Fine-Tuner – Step 3: Fine-Tune Model
**Gradio UI version**

We will:
- Verify GPU is active
- Load prepared data and GPT-2-small
- Fine-tune for 12 epochs (~90 minutes on T4)
- Save the model to disk

> ⚠️ Make sure runtime is set to **T4 GPU** before running: Runtime → Change runtime type → T4 GPU

In [4]:
import torch
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_from_disk
import os

# ── GPU check ──
if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU detected — go to Runtime → Change runtime type → T4 GPU, then re-run")

# ── Load tokenizer and datasets ──
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

tokenized_train = load_from_disk("/content/data/tokenized_train")
tokenized_val   = load_from_disk("/content/data/tokenized_val")

print(f"✅ Train chunks: {len(tokenized_train):,}")
print(f"   Val chunks:   {len(tokenized_val):,}")

# ── Load model ──
model = GPT2LMHeadModel.from_pretrained("gpt2")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ── Training arguments (tuned for 1.4 MB dataset) ──
os.makedirs("/content/models/story_voice_finetuned", exist_ok=True)

training_args = TrainingArguments(
    output_dir="/content/models/story_voice_finetuned",
    num_train_epochs=12,               # tuned for 1.4 MB
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("\n🚀 Training started — expected ~90 minutes. Don't close this tab!")
trainer.train()

trainer.save_model("/content/models/story_voice_finetuned")
tokenizer.save_pretrained("/content/models/story_voice_finetuned")
print("\n✅ Done! Model saved to /content/models/story_voice_finetuned")

✅ GPU ready: Tesla T4
✅ Train chunks: 2,372
   Val chunks:   271


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Done! Model saved to /content/models/story_voice_finetuned


### Story-Voice Fine-Tuner – Step 4: Evaluate Model

Now that the model is fine-tuned, let's see if it has learned your writing style by generating some text.

In [5]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the fine-tuned model
fine_tuned_model = GPT2LMHeadModel.from_pretrained("/content/models/story_voice_finetuned")

# Load the tokenizer (make sure it's the same one used for training)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("✅ Fine-tuned model and tokenizer loaded!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Fine-tuned model and tokenizer loaded!


In [6]:
# Define a test prompt based on your writing style
# You can change this prompt to anything you like!
prompt = "The old castle stood on a hill overlooking the"

# Encode the prompt
input_ids = tokenizer.encode(prompt, return_tensors='pt')

# Generate text
# You can adjust parameters like max_length, num_return_sequences, temperature, top_k, top_p
output = fine_tuned_model.generate(
    input_ids,
    max_length=150,           # Max length of the generated text
    num_return_sequences=1,   # Number of sequences to generate
    no_repeat_ngram_size=2,   # Avoid repeating n-grams to improve coherence
    do_sample=True,           # Use sampling
    top_k=50,                 # Sample from top_k tokens
    top_p=0.95,               # Sample from top_p probability mass
    temperature=0.7           # Control randomness
)

# Decode and print the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print("--- Generated Text ---")
print(generated_text)
print("----------------------")

print("\n💡 Does this sound like your writing? Feel free to try different prompts or generation parameters!")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- Generated Text ---
The old castle stood on a hill overlooking the river, surrounded by an endless line of demihuman soldiers.
"The siege is over!"
Ashen's eyes fell on the sky, a faint crimson glow rising above them. The sight was also the source of his confusion. His eyes hadn't been shut, so he only focused on this thing, the spear. It was his spear, and it was the same one he wielded when he was a kid. There was no spear in it, but Ashen could sense that the moment his eyes landed on something, he would instantly turn into a ghost. He had never felt so much worry and fear as he did now. But he knew the feeling. And he didn't think it would
----------------------

💡 Does this sound like your writing? Feel free to try different prompts or generation parameters!
